# Memory-Efficient Baseline Subtraction with NumPy Broadcasting

This notebook demonstrates how to perform memory-efficient baseline subtraction using NumPy's broadcasting feature. Run the cells top-to-bottom to see it in action!

In [ ]:
# First, we'll install the memory profiler library, which isn't pre-installed in Colab.
# This allows us to track memory usage precisely.
!pip install memory_profiler

# We also need to load the %memit magic command for profiling within Colab.
%load_ext memory_profiler

## Setting up our simulated sensor data and baselines

Imagine we have a massive matrix of sensor readings. Let's create a 2D NumPy array `sensor_data` with ten thousand rows and one thousand columns, filled with random floating-point numbers. This simulates 1000 sensors recording 10,000 data points each. We also need a `daily_baseline` array, a 1D array with one thousand elements, representing one baseline value for each of our 1000 sensors.

In [ ]:
import numpy as np
import pandas as pd
import os

# Define the dimensions as described in the video
rows = 10000
cols = 1000

# Create the 2D sensor data array with random floats
sensor_data = np.random.rand(rows, cols)

# Create the 1D daily baseline array with random floats
daily_baseline = np.random.rand(cols)

print(f"Shape of sensor_data: {sensor_data.shape}")
print(f"Shape of daily_baseline: {daily_baseline.shape}")
print(f"First 5 rows and 5 columns of sensor_data:\n{sensor_data[:5, :5]}")
print(f"First 10 elements of daily_baseline:\n{daily_baseline[:10]}")

## The Naive Approach's Memory Problem

If we were to try and subtract the `daily_baseline` from `sensor_data` by first expanding `daily_baseline` to match the shape of `sensor_data`, we'd be in for a rude awakening. This would involve creating a new, gigantic array, which for float64 data would consume a substantial amount of memory. As the video mentions, for a `(10000, 1000)` matrix, this would be roughly 80 GB of memory.

Let's calculate that potential memory footprint:

In [ ]:
# Calculate the memory in GB for a hypothetical expanded array
matrix_shape = (10000, 1000)
# Each float64 takes 8 bytes
memory_gb_naive = np.prod(matrix_shape) * 8 / (1024**3)
print(f'Naive approach (creating a full copy) would require approximately {memory_gb_naive:.0f} GB of additional memory.')

# Let's also see the actual memory footprint of our original arrays
baseline_size_kb = (daily_baseline.size * daily_baseline.itemsize) / 1024
sensor_data_size_mb = (sensor_data.size * sensor_data.itemsize) / (1024**2)

print(f'Current daily_baseline memory usage: {baseline_size_kb:.1f} KB')
print(f'Current sensor_data memory usage: {sensor_data_size_mb:.1f} MB')

## Memory-Efficient Subtraction using NumPy Broadcasting

This is where NumPy broadcasting shines! We can subtract the `daily_baseline` directly from `sensor_data` without explicitly creating an intermediate 80GB array. NumPy handles the 'stretching' of the `daily_baseline` across the rows implicitly, saving us from out-of-memory errors. We'll perform an *in-place* subtraction to ensure no new array is created for the result.

We'll use `%memit` to profile the memory before and after the operation, showing that no significant extra memory is allocated.

In [ ]:
# Profile memory before the subtraction
print("Memory usage BEFORE subtraction:")
%memit

# Perform the in-place subtraction using broadcasting
# NumPy automatically aligns the 1D daily_baseline with the columns of sensor_data
# and subtracts it from each row. The `[:]` ensures it's an in-place operation.
sensor_data[:] = sensor_data - daily_baseline

# Profile memory after the subtraction
print("\nMemory usage AFTER subtraction:")
%memit

print(f"\nShape of corrected sensor_data: {sensor_data.shape}")
print(f"First 5 rows and 5 columns of corrected sensor_data:\n{sensor_data[:5, :5]}")

# Verify a few elements to ensure the subtraction occurred
# For example, the first element of sensor_data[0,0] should be its original value - daily_baseline[0]
# (We can't easily show original vs new without storing original, but the fact it ran without memory issues is key)
print("\nObservation: The memory usage remains flat, confirming that no large intermediate array was created.")

## The Pandas Trap: Why Direct Assignment is Crucial

The video warns about a common pitfall: performing an operation like `df[['col1', 'col2']] - baseline` on a Pandas DataFrame might *look* like it's updating the DataFrame, but it often creates a *new* DataFrame or Series without modifying the original in place. This can lead to silently failing corrections.

While our NumPy example above used `sensor_data[:] = ...` for explicit in-place modification, if you were working with a Pandas DataFrame, you'd need to be careful with assignment. Let's illustrate the concept of the 'trap' mentioned in the video.

In [ ]:
import pandas as pd

# Create a sample DataFrame to illustrate the trap
df = pd.DataFrame({'sensor_1': [10.0, 11.0, 12.0], 'sensor_2': [20.0, 21.0, 22.0]})
print(f"Original DataFrame:\n{df}")

# Define a baseline for these two 'sensors'
baselines_df = np.array([1.0, 2.0])
print(f"\nBaselines: {baselines_df}")

# This operation computes the subtraction but returns a NEW DataFrame.
# It DOES NOT modify 'df' in place.
corrected_but_not_assigned = df[['sensor_1', 'sensor_2']] - baselines_df

print(f"\nResult of df[['sensor_1', 'sensor_2']] - baselines_df (a new object):\n{corrected_but_not_assigned}")
print(f"\nOriginal DataFrame after the operation (still unchanged!):\n{df}")

# To actually apply the correction to the DataFrame, you MUST reassign it.
df[['sensor_1', 'sensor_2']] = df[['sensor_1', 'sensor_2']] - baselines_df
print(f"\nOriginal DataFrame after correct assignment:\n{df}")

## Saving the Corrected Data

Finally, we'll save our memory-efficiently corrected `sensor_data` to a CSV file. Since `sensor_data` is a NumPy array, we'll convert it to a Pandas DataFrame first for easy CSV export.

In [ ]:
# Convert the corrected NumPy array to a Pandas DataFrame
corrected_df = pd.DataFrame(sensor_data)

# Define the filename
output_filename = 'corrected_sensor_data.csv'

# Save the DataFrame to a CSV file
corrected_df.to_csv(output_filename, index=False)

print(f"Corrected sensor data saved to '{output_filename}'")
print(f"You can find this file in the Colab file browser (left sidebar) under the root directory.")

# Display the first few rows of the saved data (after loading it back to confirm)
print("\nLoading saved data to confirm content:")
loaded_df = pd.read_csv(output_filename)
print(loaded_df.head())

## Conclusion

You've successfully performed a memory-efficient baseline subtraction on a large dataset using NumPy's broadcasting capabilities. By letting NumPy implicitly handle the array expansion, we avoided allocating gigabytes of temporary memory, a common cause of crashes in data processing.

The final corrected data is saved as `corrected_sensor_data.csv`. Remember the 'Pandas Trap': always ensure your operations are correctly assigned back to your DataFrame to avoid silently losing your changes!